===========================================================================
# Statistical Analysis and Insights
===========================================================================

The purpose of this notebook is to transform the cleaned CMS datasets into traceable statistical evidence that can be retrieved by a later Generative AI pipeline. Instead of sending raw tabular rows directly to a language model, this notebook calculates measure-specific benchmarks, facility-level deviations, performance categories, and concise evidence statements.

This design supports the project proposal's Retrieval-Augmented Generation (RAG) architecture while remaining practical on a local laptop. The statistical evidence generated here becomes the factual grounding layer used by the later prompt-engineering and summary-generation stages.

## Steps

1. Import and setup
2. Load and harmonize cleaned datasets
3. Define common facility cohort
4. Generate HCAHPS statistical insights
5. Generate HAI statistical insights
6. Generate Timely and Effective Care statistical insights
7. Build facility-level evidence documents
8. Save data

## Goal
---------------------------------------------------------------------------
Create compact, factual, measure-aware evidence documents that preserve statistical context and can be efficiently embedded, retrieved, and translated into executive, clinical, and non-technical summaries.

## Task 1: Imports and Setup
--------------------------------------------------------------------------

In [1]:
# Import necessary libraries

import pandas as pd
import numpy as np
from collections import Counter
import inspect
import json
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)


### Formatting and Repositories

In [2]:
# Formatting for printing outputs, function repository, and insight repository

def print_outputs(title = None, *outputs):

    line = "--" * 40
    double = "==" * 40

    if title:
        print(f"{double}\n{title}\n{double}")
    else:
        print(line)

    for item in outputs:

        if len(item) == 2:
            subtitle, output = item
            print_line = True
        else:
            subtitle, output, print_line = item

        if subtitle:
            print(f"{subtitle}")

        print(output)

        if print_line:
            print(line)


# Storing insights inside insights repo
insights_repository = pd.DataFrame(columns = [
    "stage",
    "section",
    "experiment",
    "parameter",
    "metric",
    "value",
    "status",
    "decision",
    "insight",
    "notes"
])


def store_insight(
    insight,
    stage = None,
    section = None,
    experiment = None,
    parameter = None,
    metric = None,
    value = None,
    status = None,
    decision = None,
    notes = None):

    global insights_repository

    new_record = pd.DataFrame([{
        "stage": stage,
        "section": section,
        "experiment": experiment,
        "parameter": parameter,
        "metric": metric,
        "value": value,
        "status": status,
        "decision": decision,
        "insight": insight,
        "notes": notes
    }])

    insights_repository = pd.concat(
        [insights_repository, new_record],
        ignore_index = True
    )

    print_outputs("New Insight Added to Repository")

    if stage:
        print(f"Stage      : {stage}")
    if section:
        print(f"Section    : {section}")
    if experiment:
        print(f"Experiment : {experiment}")
    if parameter:
        print(f"Parameter  : {parameter}")
    if metric:
        print(f"Metric     : {metric}")
    if value is not None:
        print(f"Value      : {value}")
    if status:
        print(f"Status     : {status}")
    if decision:
        print(f"Decision   : {textwrap.fill(str(decision), width = 75)}")

    print(f"\nInsight:  {textwrap.fill(str(insight), width = 75)}")

    if notes:
        print(f"\nNotes: {textwrap.fill(str(notes), width = 75)}")

    print("--" * 40)


# Create function to store information in function repository
function_repository = pd.DataFrame(
    columns = ["Function", "Description", "Input", "Output"]
)


def store_function(function, description = None, input = None, output = None):

    global function_repository

    function_name = function.__name__

    new_row = {
        "Function": function_name,
        "Description": description,
        "Input": str(inspect.signature(function)),
        "Output": output
    }

    if function_name in function_repository["Function"].values:

        for column, value in new_row.items():
            function_repository.loc[
                function_repository["Function"] == function_name,
                column
            ] = value

        print_outputs(
            f"Function {function_name} has been updated in the function_repository."
        )

    else:

        function_repository = pd.concat(
            [function_repository, pd.DataFrame([new_row])],
            ignore_index = True
        )

        print_outputs(
            f"Function '{function_name}' has been added to the function_repository."
        )


# Store repository functions
store_function(
    print_outputs,
    description = "Returns notebook outputs in a consistent organized format."
)

store_function(
    store_insight,
    description = "Stores decisions, findings, and interpretations for later reporting."
)

store_function(
    store_function,
    description = "Stores reusable functions and their signatures for traceability."
)

display(function_repository)

Function 'print_outputs' has been added to the function_repository.
Function 'store_insight' has been added to the function_repository.
Function 'store_function' has been added to the function_repository.


,Function,Description,Input,Output
0,print_outputs,Returns notebook outputs in a consistent organized format.,"(title=None, *outputs)",None
1,store_insight,"Stores decisions, findings, and interpretations for later reporting.","(insight, stage=None, section=None, experiment=None, parameter=None, metric=None, value=None, status=None, decision=...",None
2,store_function,Stores reusable functions and their signatures for traceability.,"(function, description=None, input=None, output=None)",None


### Loading Data

In [3]:
# Make sure when we are pulling in the files that the facility ID still gets read as a string
# This will prevent any potential IDs with a leading zero to be removed

survey_clean = pd.read_csv("Data/Cleaned/survey_df_clean.csv", dtype = {"Facility ID": str})
hai_clean = pd.read_csv("Data/Cleaned/ha_df_clean.csv", dtype = {"Facility ID": str})
timely_clean = pd.read_csv("Data/Cleaned/timely_df_clean.csv", dtype = {"Facility ID": str})

In [4]:
# Ensure the data types are similar across all data tables before aggregation

print(survey_clean.info())
print(hai_clean.info())
print(timely_clean.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28584 entries, 0 to 28583
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Facility ID                 28584 non-null  object
 1   Facility Name               28584 non-null  object
 2   HCAHPS Measure ID           28584 non-null  object
 3   HCAHPS Question             28584 non-null  object
 4   HCAHPS Answer Description   28584 non-null  object
 5   Patient Survey Star Rating  28584 non-null  int64 
 6   Start Date                  28584 non-null  object
 7   End Date                    28584 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.7+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54327 entries, 0 to 54326
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Facility ID           54327 non-null  object 
 1   Facility Na

## Task 2: Alignment of Datasets
------------

### Investigation

In [5]:
# Double check to make sure there are no duplicates prior to aggregation

# Check whether each facility-measure combination is unique.
survey_duplicates = survey_clean.duplicated(
    subset=["Facility ID", "HCAHPS Measure ID"],
    keep=False).sum()

hai_duplicates = hai_clean.duplicated(
    subset=["Facility ID", "Measure ID"],
    keep=False).sum()

timely_duplicates = timely_clean.duplicated(
    subset=["Facility ID", "Measure ID"],
    keep=False).sum()

# For formatting
line = '--' * 40

print("Total # of Duplicated Facility-Measure Rows Within Each Dataset")
print(line)
print("Survey duplicate facility-measures:", survey_duplicates)
print("HAI duplicate facility-measures:", hai_duplicates)
print("Timely duplicate facility-measures:", timely_duplicates)

Total # of Duplicated Facility-Measure Rows Within Each Dataset
--------------------------------------------------------------------------------
Survey duplicate facility-measures: 0
HAI duplicate facility-measures: 0
Timely duplicate facility-measures: 0


In [6]:
# Ensure proper alignment of all datasets

all_df = [survey_clean, hai_clean, timely_clean]

# Create an empty dictionary to store the facilities

facilities_list = []

for df in all_df:
    if df is survey_clean:
        df['Start Date'] = pd.to_datetime(df['Start Date'], errors = "coerce")
        df['End Date'] = pd.to_datetime(df['End Date'], errors = "coerce")

        survey_facility_list = []
        for facility in df['Facility ID']:
            survey_facility_list.append(facility)
        survey_duration = df[['Start Date', 'End Date']].value_counts()           

    elif df is hai_clean: 
        hai_facility_list = []
        for facility in df['Facility ID']:
            hai_facility_list.append(facility)
        hai_duration = df[['Start Date', 'End Date']].value_counts()           

    else:
        timely_facility_list = []
        for facility in df['Facility ID']:
            timely_facility_list.append(facility)
        timely_duration = df[['Start Date', 'End Date']].value_counts()           

print(line)
print("Unique Facilities per Data Frame: \n")
print("Number of Unique Facilities in the Survey Dataset: ", len(set(survey_facility_list)))
print("Number of Unique Facilities in the HAI Dataset: ", len(set(hai_facility_list)))
print("Number of Unique Facilities in the Timely Dataset: ", len(set(timely_facility_list)))
print(line)

print("Date Range per Data Frame: \n")
print("Unique Date Ranges for Survey Dataset: ", survey_duration)
print("Unique Date Ranges for HAI Dataset: ", hai_duration)


# Create a list of dates you want to drop

dates_to_drop = ['01/01/2024', '10/01/2024']

# Keep rows where the Start Date is NOT in that list

timely_copy = timely_clean[~timely_clean['Start Date'].isin(dates_to_drop)]

print("Unique Date Ranges for Timely Dataset: ", timely_clean[['Start Date', 'End Date']].value_counts())
print(line)

--------------------------------------------------------------------------------
Unique Facilities per Data Frame: 

Number of Unique Facilities in the Survey Dataset:  3176
Number of Unique Facilities in the HAI Dataset:  4276
Number of Unique Facilities in the Timely Dataset:  4443
--------------------------------------------------------------------------------
Date Range per Data Frame: 

Unique Date Ranges for Survey Dataset:  Start Date  End Date  
2024-07-01  2025-06-30    28584
Name: count, dtype: int64
Unique Date Ranges for HAI Dataset:  Start Date  End Date  
07/01/2024  06/30/2025    54327
Name: count, dtype: int64
Unique Date Ranges for Timely Dataset:  Start Date  End Date  
07/01/2024  06/30/2025    28952
01/01/2024  12/31/2024    24485
10/01/2024  03/31/2025     4267
Name: count, dtype: int64
--------------------------------------------------------------------------------


In [7]:
# Compile the facilities present across all three dataset

# Create sets containing unique Facility IDs
survey_facility_list = set(survey_clean["Facility ID"])
hai_facility_list = set(hai_clean["Facility ID"])
timely_facility_list = set(timely_clean["Facility ID"])

# Combine the sets
common_facilities = survey_facility_list & hai_facility_list & timely_facility_list

# Set the data type of common_facilities to a list
common_facilities = list(common_facilities)

# Print the number of common facilities
print("Total # of Facilities Present Across All Datasets: ", len(common_facilities))

Total # of Facilities Present Across All Datasets:  3042


In [8]:
# Remove the facilities that are not present across all three datasets
survey_df = survey_clean[survey_clean['Facility ID'].isin(common_facilities)]
hai_df= hai_clean[hai_clean['Facility ID'].isin(common_facilities)]
timely_df = timely_clean[timely_clean['Facility ID'].isin(common_facilities)]

# Double check that the df have the same number of unique facilities
print(line)
print("Number of Unique Facilities in survey_df: ", len(set(survey_clean['Facility ID'])))
print("Number of Unique Facilities in hai_df: ", len(set(hai_clean['Facility ID'])))
print("Number of Unique Facilities in timely_df: ", len(set(timely_clean['Facility ID'])))
print(line)

--------------------------------------------------------------------------------
Number of Unique Facilities in survey_df:  3176
Number of Unique Facilities in hai_df:  4276
Number of Unique Facilities in timely_df:  4443
--------------------------------------------------------------------------------


In [9]:
# Check to see if each facility has the same number of measures each

survey_measures = survey_df.groupby("Facility ID", as_index = False)['HCAHPS Measure ID'].nunique().rename(columns = {"HCAHPS Measure ID": "Measure Count"})
hai_measures = hai_df.groupby("Facility ID", as_index = False)['Measure ID'].nunique().rename(columns = {"Measure ID": "Measure Count"})
timely_measures = timely_df.groupby("Facility ID", as_index = False)['Measure ID'].nunique().rename(columns = {"Measure ID": "Measure Count"})

print(line)
print("Minimum # of Measures Available per Facility in survey_df:", min(survey_measures['Measure Count']))
print("Maximum # of Measures Available per Facility in survey_df:", max(survey_measures['Measure Count']))

print(line)
print("Minimum # of Measures Available per Facility in hai_df:", min(hai_measures['Measure Count']))
print("Maximum # of Measures Available per Facility in hai_df: ", max(hai_measures['Measure Count']))

print(line)
print("Minimum # of Measures Available per Facility in timely_df:", min(timely_measures['Measure Count']))
print("Maximum # of Measures Available per Facility in timely_df: ", max(timely_measures['Measure Count']))
print(line)

# Only keep facilities that have at least 80% of the total measures recorded

# No need to filter the survey_df since all facilities have nine total measures recorded

# HAI needs at least 15 out of 18 measures recorded

# Timely needs at least 8 out of 10 measures recorded

hai_measures = hai_measures[hai_measures['Measure Count'] >= 15]
timely_measures = timely_measures[timely_measures['Measure Count'] >= 8]

print("HAI Dataset: Total # of Facilities with at least 80% Measures Recorded: ", len(hai_df['Facility ID']))
print("Timely Dataset: Total # of Facilities with at least 80% Measures Recorded: ", len(timely_df['Facility ID']))
print(line)

# Update the dataframes to remove the facilities without sufficient measures

hai_df = hai_df[hai_df["Facility ID"].isin(hai_measures["Facility ID"])].copy()
timely_df = timely_df[timely_df["Facility ID"].isin(timely_measures["Facility ID"])].copy()

# Update common facilities

common_facilities = (set(survey_df["Facility ID"]) & set(hai_df["Facility ID"]) & set(timely_df["Facility ID"]))

# Print the number of facilities left

print("# of Common Facilities Remaining After Removal of Insuffiently Measured Facilities: ", len(common_facilities))
print(line)

# Update the survey_df, hai_df, and timely_df

survey_df = survey_df[survey_df['Facility ID'].isin(common_facilities)].reset_index(drop = True)
hai_df = hai_df[hai_df['Facility ID'].isin(common_facilities)].reset_index(drop = True)
timely_df = timely_df[timely_df['Facility ID'].isin(common_facilities)].reset_index(drop = True)

# Double check the number of facilities in each dataset -- should be the same across

print("Facilities in survey_df:", survey_df['Facility ID'].nunique())
print("Facilities in hai_df:", hai_df['Facility ID'].nunique())
print("Facilities in timely_df:", timely_df['Facility ID'].nunique())
print(line)

--------------------------------------------------------------------------------
Minimum # of Measures Available per Facility in survey_df: 9
Maximum # of Measures Available per Facility in survey_df: 9
--------------------------------------------------------------------------------
Minimum # of Measures Available per Facility in hai_df: 2
Maximum # of Measures Available per Facility in hai_df:  18
--------------------------------------------------------------------------------
Minimum # of Measures Available per Facility in timely_df: 1
Maximum # of Measures Available per Facility in timely_df:  23
--------------------------------------------------------------------------------
HAI Dataset: Total # of Facilities with at least 80% Measures Recorded:  45177
Timely Dataset: Total # of Facilities with at least 80% Measures Recorded:  46285
--------------------------------------------------------------------------------
# of Common Facilities Remaining After Removal of Insuffiently Measure

In [10]:
# Store insight
store_insight(insight = "After investigating the measures more deeply, there are a few items that need to be aligned before proper statistical analysis."
                        " Since there are no duplicates, no action is required. Reporting duration needs to be aligned across all the datasets."
                        "Additionally, for facility level insight, the facility should be present across all the three datasets and assess the number of"
                        " measures available for each dataset per facility (only keep facilities with at least 80% of measures recorded). Also survey column name needs to be adjusted first.", 
              decision = "Create cleaning/preprocessing function to align the datasets")

New Insight Added to Repository
Decision   : Create cleaning/preprocessing function to align the datasets

Insight:  After investigating the measures more deeply, there are a few items that
need to be aligned before proper statistical analysis. Since there are no
duplicates, no action is required. Reporting duration needs to be aligned
across all the datasets.Additionally, for facility level insight, the
facility should be present across all the three datasets and assess the
number of measures available for each dataset per facility (only keep
facilities with at least 80% of measures recorded). Also survey column name
needs to be adjusted first.
--------------------------------------------------------------------------------


### Functions
-----------

In [11]:
# Create preprocessing function to align cleaned datasets before statistical analysis

def preprocess_clean_datasets(survey, hai, timely):

    # Create copies
    survey = survey.copy()
    hai = hai.copy()
    timely = timely.copy()

    # Rename survey column
    survey = survey.rename(columns = {
        "HCAHPS Measure ID": "Measure ID",
        "HCAHPS Question": "Measure Name",
        "Patient Survey Star Rating": "Score"})

    # Pull the reporting data frame from each df
    for dataframe in [survey, hai, timely]:
        dataframe[["Start Date", "End Date"]] = dataframe[["Start Date", "End Date"]].apply(pd.to_datetime)

    dates = ["Start Date", "End Date"]

    # Pull the shared periods
    shared_periods = (survey[dates].drop_duplicates()
                      .merge(hai[dates].drop_duplicates(), on = dates)
                      .merge(timely[dates].drop_duplicates(), on = dates))

    # Count the shared periods
    combined_dates = pd.concat([survey[dates], hai[dates], timely[dates]], ignore_index = True)
    shared_counts = shared_periods.merge(combined_dates.value_counts().reset_index(name = "Count"), on = dates)

    common_period = shared_counts.sort_values("Count", ascending = False).iloc[0]
    common_start, common_end = common_period["Start Date"], common_period["End Date"]

    # Filter the dataframe to only include the common reporting periods
    survey = survey[(survey["Start Date"] == common_start) & (survey["End Date"] == common_end)].copy()
    hai = hai[(hai["Start Date"] == common_start) & (hai["End Date"] == common_end)].copy()
    timely = timely[(timely["Start Date"] == common_start) & (timely["End Date"] == common_end)].copy()

    shared_facilities = set(survey["Facility ID"]) & set(hai["Facility ID"]) & set(timely["Facility ID"])

    survey = survey[survey["Facility ID"].isin(shared_facilities)].copy()
    hai = hai[hai["Facility ID"].isin(shared_facilities)].copy()
    timely = timely[timely["Facility ID"].isin(shared_facilities)].copy()

    survey_measures = survey.groupby("Facility ID")["Measure ID"].nunique().reset_index(name = "Survey Measures")
    hai_measures = hai.groupby("Facility ID")["Measure ID"].nunique().reset_index(name = "HAI Measures")
    timely_measures = timely.groupby("Facility ID")["Measure ID"].nunique().reset_index(name = "Timely Measures")

    facility_measure_counts = (survey_measures.merge(hai_measures, on = "Facility ID")
                               .merge(timely_measures, on = "Facility ID"))

    print(f"Selected reporting period: {common_start.date()} to {common_end.date()}")
    print("Shared facilities:", len(shared_facilities))

    return survey, hai, timely, facility_measure_counts

# Store function
store_function(
    function = preprocess_clean_datasets,
    description = "Aligns reporting periods, shared facilities, and measure availability across datasets.")

Function 'preprocess_clean_datasets' has been added to the function_repository.


In [12]:
# Implementation

survey_clean, hai_clean, timely_clean, facility_measure_counts = preprocess_clean_datasets(survey_clean, hai_clean, timely_clean)

print_outputs("Applying the preprocessing function to align reporting periods, identify shared facilities, and number of measures per df for every facility",
              ("survey_clean", survey_clean.head(2), True),
              ("hai_clean", hai_clean.head(2), True), 
              ("timely_clean", timely_clean.head(2), True),
              ("Facility Measure Count", facility_measure_counts))


Selected reporting period: 2024-07-01 to 2025-06-30
Shared facilities: 2957
Applying the preprocessing function to align reporting periods, identify shared facilities, and number of measures per df for every facility
survey_clean
  Facility ID                    Facility Name            Measure ID  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER  H_COMP_1_STAR_RATING   
1      010001  SOUTHEAST HEALTH MEDICAL CENTER  H_COMP_2_STAR_RATING   

                         Measure Name           HCAHPS Answer Description  \
0   Nurse communication - star rating   Nurse communication - star rating   
1  Doctor communication - star rating  Doctor communication - star rating   

   Score Start Date   End Date  
0      3 2024-07-01 2025-06-30  
1      4 2024-07-01 2025-06-30  
--------------------------------------------------------------------------------
hai_clean
  Facility ID                    Facility Name       Measure ID  \
0      010001  SOUTHEAST HEALTH MEDICAL CENTER       HAI_1_DOPC 

In [13]:
store_insight(
    stage = "Alignment of Datasets",
    section = "Implementation",
    metric = "Common Facilities",
    value = len(facility_measure_counts['Facility ID']),
    insight = (f"The integrated analysis contains {len(facility_measure_counts['Facility ID']):,} hospitals with evidence available in all three CMS quality domains. Does not filter out facilities with underreported (less than 80%) measures")
)

New Insight Added to Repository
Stage      : Alignment of Datasets
Section    : Implementation
Metric     : Common Facilities
Value      : 2957

Insight:  The integrated analysis contains 2,957 hospitals with evidence available in
all three CMS quality domains. Does not filter out facilities with
underreported (less than 80%) measures
--------------------------------------------------------------------------------


In [14]:
# Create function to retain facilities with at least 80% measure availability across all datasets

def filter_facility_measure_availability(survey, hai, timely, facility_measure_counts, threshold = 0.80):

    survey = survey.copy()
    hai = hai.copy()
    timely = timely.copy()
    counts = facility_measure_counts.copy()

    measure_columns = ["Survey Measures", "HAI Measures", "Timely Measures"]

    for column in measure_columns:
        counts[f"{column} Required"] = np.ceil(counts[column].max() * threshold).astype(int)

    eligible = counts[
        (counts["Survey Measures"] >= counts["Survey Measures Required"]) &
        (counts["HAI Measures"] >= counts["HAI Measures Required"]) &
        (counts["Timely Measures"] >= counts["Timely Measures Required"])
    ]["Facility ID"]

    survey = survey[survey["Facility ID"].isin(eligible)].copy()
    hai = hai[hai["Facility ID"].isin(eligible)].copy()
    timely = timely[timely["Facility ID"].isin(eligible)].copy()

    facilities_available = f"Facilities retained with at least {threshold:.0%} measure availability: {eligible.nunique()}"

    return survey, hai, timely, facilities_available


store_function(
    function = filter_facility_measure_availability,
    description = "Retains facilities with at least 80% measure availability across all datasets."
)

Function 'filter_facility_measure_availability' has been added to the function_repository.


In [15]:
# Implementation: filter_facility_measure_availability

survey_clean, hai_clean, timely_clean, facilities_available = filter_facility_measure_availability(survey_clean, hai_clean, timely_clean, facility_measure_counts)

print_outputs("Applying the preprocessing function to filter out facilities with underreported measures",
              ("survey_clean", survey_clean.head(2), True),
              ("hai_clean", hai_clean.head(2), True), 
              ("timely_clean", timely_clean.head(2), True),
              (None, facilities_available))

Applying the preprocessing function to filter out facilities with underreported measures
survey_clean
   Facility ID             Facility Name            Measure ID  \
9       010005  MARSHALL MEDICAL CENTERS  H_COMP_1_STAR_RATING   
10      010005  MARSHALL MEDICAL CENTERS  H_COMP_2_STAR_RATING   

                          Measure Name           HCAHPS Answer Description  \
9    Nurse communication - star rating   Nurse communication - star rating   
10  Doctor communication - star rating  Doctor communication - star rating   

    Score Start Date   End Date  
9       3 2024-07-01 2025-06-30  
10      4 2024-07-01 2025-06-30  
--------------------------------------------------------------------------------
hai_clean
   Facility ID             Facility Name       Measure ID  \
17      010005  MARSHALL MEDICAL CENTERS       HAI_1_DOPC   
18      010005  MARSHALL MEDICAL CENTERS  HAI_1_NUMERATOR   

                                                                          Measure Name 

In [16]:
# Create function to standardize measure names and units across datasets

def standardize_measures_and_units(survey, hai, timely):

    survey = survey.copy()
    hai = hai.copy()
    timely = timely.copy()

    # Patient Survey: shorten Measure Name and assign units
    survey["Measure Name"] = (survey["Measure Name"].astype(str)
                              .str.replace(r"\s*-\s*star rating$", "", case = False, regex = True)
                              .str.replace(r"\s*star rating$", "", case = False, regex = True)
                              .str.strip())

    survey["Units"] = "Stars"

    # HAI: standardize infection names using Measure ID family
    hai_infection_types = {
        "HAI_1": "CLABSI (ICU + select Wards)",
        "HAI_2": "CAUTI (ICU + select Wards)",
        "HAI_3": "SSI (Colon Surgery)",
        "HAI_4": "SSI (Abdominal Hysterectomy)",
        "HAI_5": "MRSA Bacteremia",
        "HAI_6": "Clostridium Difficile"
    }

    for measure_prefix, infection_type in hai_infection_types.items():
        mask = hai["Measure ID"].str.startswith(measure_prefix, na = False)
        hai.loc[mask, "Measure Name"] = infection_type

    # HAI: assign units using Measure ID
    hai["Units"] = None

    hai.loc[hai["Measure ID"].str.endswith("_SIR", na = False), "Units"] = \
        "Ratio (Observed/Predicted Cases)"

    hai.loc[hai["Measure ID"].str.endswith("_NUMERATOR", na = False), "Units"] = "Cases"

    hai.loc[hai["Measure ID"].eq("HAI_1_DOPC"), "Units"] = "Device Days"
    hai.loc[hai["Measure ID"].eq("HAI_2_DOPC"), "Units"] = "Catheter Days"
    hai.loc[hai["Measure ID"].isin(["HAI_3_DOPC", "HAI_4_DOPC"]), "Units"] = "Procedures"
    hai.loc[hai["Measure ID"].isin(["HAI_5_DOPC", "HAI_6_DOPC"]), "Units"] = "Patient Days"

    # Timely and Effective Care: standardize names and units using Measure ID
    timely_measure_context = {
        "OP_18a": {"name": "ED Time - All Patients", "units": "Minutes"},
        "OP_18b": {"name": "ED Time - General Patients", "units": "Minutes"},
        "OP_18c": {"name": "ED Time - Psychiatric Patients", "units": "Minutes"},
        "OP_18d": {"name": "ED Time - Transfer Patients", "units": "Minutes"},
        "OP_23": {"name": "Head CT Results Within 45 Minutes", "units": "Percent"},
        "SEP_1": {"name": "Severe Sepsis and Septic Shock Care", "units": "Percent"},
        "SEP_SH_3HR": {"name": "Septic Shock 3-Hour Bundle", "units": "Percent"},
        "SEP_SH_6HR": {"name": "Septic Shock 6-Hour Bundle", "units": "Percent"},
        "SEV_SEP_3HR": {"name": "Severe Sepsis 3-Hour Bundle", "units": "Percent"},
        "SEV_SEP_6HR": {"name": "Severe Sepsis 6-Hour Bundle", "units": "Percent"}
    }

    timely["Measure Name"] = timely["Measure ID"].map(
        {measure_id: context["name"] for measure_id, context in timely_measure_context.items()}
    )

    timely["Units"] = timely["Measure ID"].map(
        {measure_id: context["units"] for measure_id, context in timely_measure_context.items()}
    )

    return survey, hai, timely


store_function(
    function = standardize_measures_and_units,
    description = "Standardizes measure names and units across all datasets."
)

Function 'standardize_measures_and_units' has been added to the function_repository.


In [17]:
# Implementation

survey_clean, hai_clean, timely_clean = standardize_measures_and_units(survey_clean, hai_clean, timely_clean)


In [18]:
# Create function to add measure interpretation direction

def add_measure_direction(survey, hai, timely):

    survey = survey.copy()
    hai = hai.copy()
    timely = timely.copy()

    survey["Direction"] = "Higher is Better"

    hai["Direction"] = "Context Only"
    hai.loc[hai["Measure ID"].str.endswith("_SIR", na = False), "Direction"] = "Lower is Better"

    timely_direction = {
        "OP_18a": "Lower is Better",
        "OP_18b": "Lower is Better",
        "OP_18c": "Lower is Better",
        "OP_18d": "Lower is Better",
        "OP_23": "Higher is Better",
        "SEP_1": "Higher is Better",
        "SEP_SH_3HR": "Higher is Better",
        "SEP_SH_6HR": "Higher is Better",
        "SEV_SEP_3HR": "Higher is Better",
        "SEV_SEP_6HR": "Higher is Better"
    }

    timely["Direction"] = timely["Measure ID"].map(timely_direction)

    return survey, hai, timely


store_function(
    function = add_measure_direction,
    description = "Adds measure-specific interpretation direction across datasets."
)

Function 'add_measure_direction' has been added to the function_repository.


In [19]:
# Implementation
survey_clean, hai_clean, timely_clean = add_measure_direction(survey_clean, hai_clean, timely_clean)

## Task 3: Statistical Insight for HCAHPS Patient Survey
-----------

#### Investigation

In [20]:
# Assess new shape of survey clean -- should be 1646 facilities * 8 measures = 14814 rows

print("Final Shape of survey_clean: ", survey_clean.shape)
print(survey_clean.columns)

Final Shape of survey_clean:  (14814, 10)
Index(['Facility ID', 'Facility Name', 'Measure ID', 'Measure Name',
       'HCAHPS Answer Description', 'Score', 'Start Date', 'End Date', 'Units',
       'Direction'],
      dtype='object')


In [21]:
# Compare numerical and non-numerical Patient Survey Star Rating values

survey_numeric = pd.to_numeric(survey_clean["Score"], errors = "coerce")

print("Numerical:", survey_numeric.notna().sum())
print("Non-Numerical:", survey_numeric.isna().sum())

Numerical: 14814
Non-Numerical: 0


In [22]:
# Quickly assess skewness by measure

survey_skew = survey_clean.groupby("Measure ID")["Score"].skew()

display(survey_skew)


Measure ID
H_CLEAN_STAR_RATING        -0.325364
H_COMP_1_STAR_RATING       -0.016375
H_COMP_2_STAR_RATING       -0.080348
H_COMP_5_STAR_RATING        0.208134
H_COMP_6_STAR_RATING       -0.280397
H_HSP_RATING_STAR_RATING   -0.272111
H_QUIET_STAR_RATING        -0.078704
H_RECMND_STAR_RATING       -0.265795
H_STAR_RATING              -0.187825
Name: Score, dtype: float64

In [23]:
# Store insight
store_insight(insight = "There are no non-numerical survey rating values and the survey data is not too skewed since the measures all are within the + or - .5 range. Therefore when calculating benchmarks, aggregated mean should work better than median.")

New Insight Added to Repository

Insight:  There are no non-numerical survey rating values and the survey data is not
too skewed since the measures all are within the + or - .5 range. Therefore
when calculating benchmarks, aggregated mean should work better than
median.
--------------------------------------------------------------------------------


### Function

In [24]:
# Create function to calculate measure-specific cohort benchmarks for patient surveys

def calculate_measure_benchmarks(dataframe, score_column, measure_id_column = "Measure ID",
                                 measure_name_column = "Measure Name"):

    required = [score_column, measure_id_column, measure_name_column]
    missing = [column for column in required if column not in dataframe.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    numeric = dataframe.copy()
    numeric[score_column] = pd.to_numeric(numeric[score_column], errors = "coerce")
    numeric = numeric.dropna(subset = [score_column])

    benchmarks = (numeric.groupby([measure_id_column, measure_name_column], as_index = False)
                  .agg(Benchmark_Cohort_Mean = (score_column, "mean"))
                  .rename(columns = {"Benchmark_Cohort_Mean": "Benchmark Cohort Mean"}))

    return benchmarks


store_function(
    function = calculate_measure_benchmarks,
    description = "Calculates measure-specific cohort mean from numeric score values."
)

Function 'calculate_measure_benchmarks' has been added to the function_repository.


In [25]:
# Create function to compare facility scores with measure-specific cohort benchmarks

def add_facility_comparison(dataframe, benchmarks, score_column):

    result = dataframe.copy()
    result[score_column] = pd.to_numeric(result[score_column], errors = "coerce")
    result = result.dropna(subset = [score_column])

    result = result.merge(benchmarks, on = ["Measure ID", "Measure Name"], how = "left")

    result["Difference From Cohort Mean"] = result[score_column] - result["Benchmark Cohort Mean"]

    result["Cohort Std"] = result.groupby("Measure ID")[score_column].transform("std")
    threshold = result["Cohort Std"] * 0.5

    result["Relative Position to Cohort"] = np.select(
        [result["Difference From Cohort Mean"] > threshold,
         result["Difference From Cohort Mean"] < -threshold],
        ["Above Cohort Mean", "Below Cohort Mean"],
        default = "Similar to Cohort Mean"
    )

    if "HCAHPS Answer Description" in result.columns:
        result = result.drop(columns = ["HCAHPS Answer Description"])

    return result


store_function(
    function = add_facility_comparison,
    description = "Compares facility scores with cohort benchmarks and retains cohort variability."
)

Function 'add_facility_comparison' has been added to the function_repository.


#### Implementation

In [26]:
# Implementation: calculate_measure_benchmarks

survey_benchmarks = calculate_measure_benchmarks(survey_clean, score_column = "Score")

survey_analysis = add_facility_comparison(survey_clean, benchmarks = survey_benchmarks, score_column = "Score")

display(survey_benchmarks)
display(survey_analysis)

,Measure ID,Measure Name,Benchmark Cohort Mean
0,H_CLEAN_STAR_RATING,Cleanliness,3.001215
1,H_COMP_1_STAR_RATING,Nurse communication,3.027947
2,H_COMP_2_STAR_RATING,Doctor communication,2.969016
3,H_COMP_5_STAR_RATING,Communication about medicines,2.247874
4,H_COMP_6_STAR_RATING,Discharge information,3.095990
5,H_HSP_RATING_STAR_RATING,Overall hospital rating,2.989064
6,H_QUIET_STAR_RATING,Quietness,2.777035
7,H_RECMND_STAR_RATING,Recommend hospital,3.411300
8,H_STAR_RATING,Summary,2.962333


,Facility ID,Facility Name,Measure ID,Measure Name,Score,Start Date,End Date,Units,Direction,Benchmark Cohort Mean,Difference From Cohort Mean,Cohort Std,Relative Position to Cohort
0,010005,MARSHALL MEDICAL CENTERS,H_COMP_1_STAR_RATING,Nurse communication,3,2024-07-01,2025-06-30,Stars,Higher is Better,3.027947,-0.027947,0.910164,Similar to Cohort Mean
1,010005,MARSHALL MEDICAL CENTERS,H_COMP_2_STAR_RATING,Doctor communication,4,2024-07-01,2025-06-30,Stars,Higher is Better,2.969016,1.030984,0.676761,Above Cohort Mean
2,010005,MARSHALL MEDICAL CENTERS,H_COMP_5_STAR_RATING,Communication about medicines,2,2024-07-01,2025-06-30,Stars,Higher is Better,2.247874,-0.247874,0.684553,Similar to Cohort Mean
3,010005,MARSHALL MEDICAL CENTERS,H_COMP_6_STAR_RATING,Discharge information,3,2024-07-01,2025-06-30,Stars,Higher is Better,3.095990,-0.095990,0.897424,Similar to Cohort Mean
4,010005,MARSHALL MEDICAL CENTERS,H_CLEAN_STAR_RATING,Cleanliness,3,2024-07-01,2025-06-30,Stars,Higher is Better,3.001215,-0.001215,0.878655,Similar to Cohort Mean
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14809,670260,TEXAS HEALTH HOSPITAL FRISCO,H_CLEAN_STAR_RATING,Cleanliness,4,2024-07-01,2025-06-30,Stars,Higher is Better,3.001215,0.998785,0.878655,Above Cohort Mean
14810,670260,TEXAS HEALTH HOSPITAL FRISCO,H_QUIET_STAR_RATING,Quietness,5,2024-07-01,2025-06-30,Stars,Higher is Better,2.777035,2.222965,0.881134,Above Cohort Mean
14811,670260,TEXAS HEALTH HOSPITAL FRISCO,H_HSP_RATING_STAR_RATING,Overall hospital rating,4,2024-07-01,2025-06-30,Stars,Higher is Better,2.989064,1.010936,0.881351,Above Cohort Mean
14812,670260,TEXAS HEALTH HOSPITAL FRISCO,H_RECMND_STAR_RATING,Recommend hospital,5,2024-07-01,2025-06-30,Stars,Higher is Better,3.411300,1.588700,1.035113,Above Cohort Mean


In [27]:
print(survey_analysis['Measure Name'].value_counts())

Measure Name
Nurse communication              1646
Doctor communication             1646
Communication about medicines    1646
Discharge information            1646
Cleanliness                      1646
Quietness                        1646
Overall hospital rating          1646
Recommend hospital               1646
Summary                          1646
Name: count, dtype: int64


## Task 4: Statistical Insight for Healthcare Acquired Infections (HAIs)
------------

#### Investigation

In [28]:
# Pull in the cleaned df
print(line)
print("HAI Data Frame Columns: ", hai_clean.columns.tolist())
print(line)

# Make sure the number of unique facilities matches the previous df -- should be 1646
print("Total # of Unique Facilities: ", hai_clean['Facility ID'].nunique())
print(line)

# Double check for null values
print("Null Values Within hai_clean: ", hai_clean.isnull().sum())
print(line)

# Assess how many measures exist
print("Total # of Unique Measures Captured: ", hai_clean[['Measure ID', 'Measure Name']].value_counts())
print(line)

--------------------------------------------------------------------------------
HAI Data Frame Columns:  ['Facility ID', 'Facility Name', 'Measure ID', 'Measure Name', 'Compared to National', 'Score', 'Start Date', 'End Date', 'Units', 'Direction']
--------------------------------------------------------------------------------
Total # of Unique Facilities:  1646
--------------------------------------------------------------------------------
Null Values Within hai_clean:  Facility ID             0
Facility Name           0
Measure ID              0
Measure Name            0
Compared to National    0
Score                   0
Start Date              0
End Date                0
Units                   0
Direction               0
dtype: int64
--------------------------------------------------------------------------------
Total # of Unique Measures Captured:  Measure ID       Measure Name                
HAI_1_DOPC       CLABSI (ICU + select Wards)     1646
HAI_3_NUMERATOR  SSI (Colon S

#### Function

#### Implementation

In [29]:
# Implementation: calculate_measure_benchmarks

hai_benchmarks = calculate_measure_benchmarks(hai_clean, score_column = "Score")

hai_analysis = add_facility_comparison(hai_clean, benchmarks = hai_benchmarks, score_column = "Score")

display(hai_benchmarks)
display(hai_analysis)


,Measure ID,Measure Name,Benchmark Cohort Mean
0,HAI_1_DOPC,CLABSI (ICU + select Wards),8814.777035
1,HAI_1_NUMERATOR,CLABSI (ICU + select Wards),5.188943
2,HAI_1_SIR,CLABSI (ICU + select Wards),0.552722
3,HAI_2_DOPC,CAUTI (ICU + select Wards),8182.139733
4,HAI_2_NUMERATOR,CAUTI (ICU + select Wards),4.915553
5,HAI_2_SIR,CAUTI (ICU + select Wards),0.497320
6,HAI_3_DOPC,SSI (Colon Surgery),171.579587
7,HAI_3_NUMERATOR,SSI (Colon Surgery),3.998785
8,HAI_3_SIR,SSI (Colon Surgery),0.834905
9,HAI_4_DOPC,SSI (Abdominal Hysterectomy),112.827757


,Facility ID,Facility Name,Measure ID,Measure Name,Compared to National,Score,Start Date,End Date,Units,Direction,Benchmark Cohort Mean,Difference From Cohort Mean,Cohort Std,Relative Position to Cohort
0,010005,MARSHALL MEDICAL CENTERS,HAI_1_DOPC,CLABSI (ICU + select Wards),No Different than National Benchmark,4837.000,2024-07-01,2025-06-30,Device Days,Context Only,8814.777035,-3977.777035,11077.999577,Similar to Cohort Mean
1,010005,MARSHALL MEDICAL CENTERS,HAI_1_NUMERATOR,CLABSI (ICU + select Wards),No Different than National Benchmark,5.000,2024-07-01,2025-06-30,Cases,Context Only,5.188943,-0.188943,8.720037,Similar to Cohort Mean
2,010005,MARSHALL MEDICAL CENTERS,HAI_1_SIR,CLABSI (ICU + select Wards),No Different than National Benchmark,1.694,2024-07-01,2025-06-30,Ratio (Observed/Predicted Cases),Lower is Better,0.552722,1.141278,0.511284,Above Cohort Mean
3,010005,MARSHALL MEDICAL CENTERS,HAI_2_DOPC,CAUTI (ICU + select Wards),No Different than National Benchmark,6302.000,2024-07-01,2025-06-30,Catheter Days,Context Only,8182.139733,-1880.139733,7619.023414,Similar to Cohort Mean
4,010005,MARSHALL MEDICAL CENTERS,HAI_2_NUMERATOR,CAUTI (ICU + select Wards),No Different than National Benchmark,5.000,2024-07-01,2025-06-30,Cases,Context Only,4.915553,0.084447,7.003571,Similar to Cohort Mean
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27973,670260,TEXAS HEALTH HOSPITAL FRISCO,HAI_5_DOPC,MRSA Bacteremia,Not Available,27423.000,2024-07-01,2025-06-30,Patient Days,Context Only,83179.202916,-55756.202916,79071.348119,Below Cohort Mean
27974,670260,TEXAS HEALTH HOSPITAL FRISCO,HAI_5_NUMERATOR,MRSA Bacteremia,Not Available,2.000,2024-07-01,2025-06-30,Cases,Context Only,3.893074,-1.893074,5.603038,Similar to Cohort Mean
27975,670260,TEXAS HEALTH HOSPITAL FRISCO,HAI_6_DOPC,Clostridium Difficile,Better than the National Benchmark,23537.000,2024-07-01,2025-06-30,Patient Days,Context Only,77255.661604,-53718.661604,72197.385933,Below Cohort Mean
27976,670260,TEXAS HEALTH HOSPITAL FRISCO,HAI_6_NUMERATOR,Clostridium Difficile,Better than the National Benchmark,0.000,2024-07-01,2025-06-30,Cases,Context Only,14.475091,-14.475091,19.393956,Below Cohort Mean


## Task 5: Statistical Insight for Timely and Effective Care
------------

#### Function

In [30]:
# Implementation: calculate_measure_benchmarks

timely_benchmarks = calculate_measure_benchmarks(timely_clean, score_column = "Score")

timely_analysis = add_facility_comparison(timely_clean, benchmarks = timely_benchmarks, score_column = "Score")

display(timely_benchmarks)
display(timely_analysis)

,Measure ID,Measure Name,Benchmark Cohort Mean
0,OP_18a,ED Time - All Patients,196.201094
1,OP_18b,ED Time - General Patients,191.648238
2,OP_18c,ED Time - Psychiatric Patients,319.042895
3,OP_18d,ED Time - Transfer Patients,357.589878
4,OP_23,Head CT Results Within 45 Minutes,71.460930
5,SEP_1,Severe Sepsis and Septic Shock Care,62.751519
6,SEP_SH_3HR,Septic Shock 3-Hour Bundle,71.205471
7,SEP_SH_6HR,Septic Shock 6-Hour Bundle,85.814148
8,SEV_SEP_3HR,Severe Sepsis 3-Hour Bundle,79.537060
9,SEV_SEP_6HR,Severe Sepsis 6-Hour Bundle,91.854192


,Facility ID,Facility Name,State,Condition,Measure ID,Measure Name,Score,Start Date,End Date,Units,Direction,Benchmark Cohort Mean,Difference From Cohort Mean,Cohort Std,Relative Position to Cohort
0,010005,MARSHALL MEDICAL CENTERS,AL,Emergency Department,OP_18a,ED Time - All Patients,145,2024-07-01,2025-06-30,Minutes,Lower is Better,196.201094,-51.201094,47.609751,Below Cohort Mean
1,010005,MARSHALL MEDICAL CENTERS,AL,Emergency Department,OP_18b,ED Time - General Patients,141,2024-07-01,2025-06-30,Minutes,Lower is Better,191.648238,-50.648238,46.720597,Below Cohort Mean
2,010005,MARSHALL MEDICAL CENTERS,AL,Emergency Department,OP_18c,ED Time - Psychiatric Patients,280,2024-07-01,2025-06-30,Minutes,Lower is Better,319.042895,-39.042895,194.234543,Similar to Cohort Mean
3,010005,MARSHALL MEDICAL CENTERS,AL,Emergency Department,OP_18d,ED Time - Transfer Patients,314,2024-07-01,2025-06-30,Minutes,Lower is Better,357.589878,-43.589878,142.467863,Similar to Cohort Mean
4,010005,MARSHALL MEDICAL CENTERS,AL,Sepsis Care,SEP_1,Severe Sepsis and Septic Shock Care,73,2024-07-01,2025-06-30,Percent,Higher is Better,62.751519,10.248481,14.264104,Above Cohort Mean
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14501,670260,TEXAS HEALTH HOSPITAL FRISCO,TX,Sepsis Care,SEP_1,Severe Sepsis and Septic Shock Care,78,2024-07-01,2025-06-30,Percent,Higher is Better,62.751519,15.248481,14.264104,Above Cohort Mean
14502,670260,TEXAS HEALTH HOSPITAL FRISCO,TX,Sepsis Care,SEP_SH_3HR,Septic Shock 3-Hour Bundle,84,2024-07-01,2025-06-30,Percent,Higher is Better,71.205471,12.794529,15.301006,Above Cohort Mean
14503,670260,TEXAS HEALTH HOSPITAL FRISCO,TX,Sepsis Care,SEP_SH_6HR,Septic Shock 6-Hour Bundle,95,2024-07-01,2025-06-30,Percent,Higher is Better,85.814148,9.185852,12.458768,Above Cohort Mean
14504,670260,TEXAS HEALTH HOSPITAL FRISCO,TX,Sepsis Care,SEV_SEP_3HR,Severe Sepsis 3-Hour Bundle,87,2024-07-01,2025-06-30,Percent,Higher is Better,79.537060,7.462940,9.331622,Above Cohort Mean


## Task 7: Build Facility-Level Documents
----------------------------------------

#### Function

In [31]:
# Create function to build narrative-ready evidence repository

def build_evidence_repository(survey_analysis, hai_analysis, timely_analysis, decimals = 2):

    survey = survey_analysis.copy()
    hai = hai_analysis.copy()
    timely = timely_analysis.copy()

    survey["Domain"] = "Patient Survey"
    hai["Domain"] = "Healthcare-Associated Infections"
    timely["Domain"] = "Timely and Effective Care"

    combined = pd.concat([survey, hai, timely], ignore_index = True)

    numeric_columns = ["Score", "Benchmark Cohort Mean", "Difference From Cohort Mean"]

    for column in numeric_columns:
        combined[column] = pd.to_numeric(combined[column], errors = "coerce").round(decimals)

    evidence_columns = [
        "Facility ID",
        "Facility Name",
        "Domain",
        "Condition",
        "Measure ID",
        "Measure Name",
        "Score",
        "Units",
        "Direction",
        "Benchmark Cohort Mean",
        "Cohort Std",
        "Difference From Cohort Mean",
        "Relative Position to Cohort",
        "Compared to National"
    ]

    evidence_repository = combined[evidence_columns].copy()

    return evidence_repository


store_function(
    function = build_evidence_repository,
    description = "Combines statistical results and measure context into a narrative-ready evidence repository."
)

Function 'build_evidence_repository' has been added to the function_repository.


In [32]:
# Implementation: build narrative-ready evidence repository

evidence_repository = build_evidence_repository(
    survey_analysis,
    hai_analysis,
    timely_analysis,
    decimals = 2
)

display(evidence_repository.head())

,Facility ID,Facility Name,Domain,Condition,Measure ID,Measure Name,Score,Units,Direction,Benchmark Cohort Mean,Cohort Std,Difference From Cohort Mean,Relative Position to Cohort,Compared to National
0,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_1_STAR_RATING,Nurse communication,3.0,Stars,Higher is Better,3.03,0.910164,-0.03,Similar to Cohort Mean,NaN
1,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_2_STAR_RATING,Doctor communication,4.0,Stars,Higher is Better,2.97,0.676761,1.03,Above Cohort Mean,NaN
2,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_5_STAR_RATING,Communication about medicines,2.0,Stars,Higher is Better,2.25,0.684553,-0.25,Similar to Cohort Mean,NaN
3,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_COMP_6_STAR_RATING,Discharge information,3.0,Stars,Higher is Better,3.10,0.897424,-0.10,Similar to Cohort Mean,NaN
4,010005,MARSHALL MEDICAL CENTERS,Patient Survey,NaN,H_CLEAN_STAR_RATING,Cleanliness,3.0,Stars,Higher is Better,3.00,0.878655,-0.00,Similar to Cohort Mean,NaN


## Save Data

In [33]:
# Save outputs

from pathlib import Path

statistics_folder = Path("Data/Statistical Analysis")

statistics_folder.mkdir(parents = True, exist_ok = True)

survey_benchmarks.to_csv(statistics_folder / "survey_measure_benchmarks.csv", index = False)
hai_benchmarks.to_csv(statistics_folder / "hai_measure_benchmarks.csv", index = False)
timely_benchmarks.to_csv(statistics_folder / "timely_measure_benchmarks.csv", index = False)

survey_analysis.to_csv(statistics_folder / "survey_statistical_analysis.csv", index = False)
hai_analysis.to_csv(statistics_folder / "hai_statistical_analysis.csv", index = False)
timely_analysis.to_csv(statistics_folder / "timely_statistical_analysis.csv", index = False)

evidence_repository.to_csv(statistics_folder / "evidence_repository.csv", index = False)

print(line)
print("Statistical analysis and evidence repository saved successfully.")
print("Output Folder: ", "Data/Statistical Analysis")
print(line)

--------------------------------------------------------------------------------
Statistical analysis and evidence repository saved successfully.
Output Folder:  Data/Statistical Analysis
--------------------------------------------------------------------------------
